# Assignment No. 3
## Title:

Text Preprocessing and TF-IDF Representation

## Problem Statement:

Perform text cleaning, lemmatization, stop word removal, and label encoding. Create feature representation using TF-IDF and save outputs.

In [1]:
# Install if needed
# !pip install nltk pandas scikit-learn

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Anjali/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Anjali/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
df = pd.read_pickle('News_dataset.pickle')

df.head()

,File_Name,Content,Category,Complete_Filename,id,News_length
0,001.txt,Ad sales boost Time Warner profit\r\n\r\nQuart...,business,001.txt-business,1,2569
1,002.txt,Dollar gains on Greenspan speech\r\n\r\nThe do...,business,002.txt-business,1,2257
2,003.txt,Yukos unit buyer faces loan claim\r\n\r\nThe o...,business,003.txt-business,1,1557
3,004.txt,High fuel prices hit BA's profits\r\n\r\nBriti...,business,004.txt-business,1,2421
4,005.txt,Pernod takeover talk lifts Domecq\r\n\r\nShare...,business,005.txt-business,1,1575


In [5]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Lowercase
    text = text.lower()
    
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    
    # Tokenize
    words = text.split()
    
    # Remove stopwords + Lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)

In [6]:
df['clean_text'] = df['Content'].apply(preprocess)

df[['Content', 'clean_text']].head()

,Content,clean_text
0,Ad sales boost Time Warner profit\r\n\r\nQuart...,ad sale boost time warner profit quarterly pro...
1,Dollar gains on Greenspan speech\r\n\r\nThe do...,dollar gain greenspan speech dollar hit highes...
2,Yukos unit buyer faces loan claim\r\n\r\nThe o...,yukos unit buyer face loan claim owner embattl...
3,High fuel prices hit BA's profits\r\n\r\nBriti...,high fuel price hit ba profit british airway b...
4,Pernod takeover talk lifts Domecq\r\n\r\nShare...,pernod takeover talk lift domecq share uk drin...


In [8]:
category_codes = {
    'business': 0,
    'entertainment': 1,
    'politics': 2,
    'sport': 3,
    'tech': 4
}

df['Category_Code'] = df['Category'].map(category_codes)

df[['Category', 'Category_Code']].head()

,Category,Category_Code
0,business,0
1,business,0
2,business,0
3,business,0
4,business,0


In [9]:
tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1,2),
    min_df=10,
    max_df=1.0
)

X = tfidf.fit_transform(df['clean_text']).toarray()

print("Shape of TF-IDF matrix:", X.shape)

Shape of TF-IDF matrix: (2225, 300)


In [10]:
feature_names = tfidf.get_feature_names_out()

print("Top Features:\n", feature_names[:20])

Top Features:
 ['10' '2003' '2004' '2005' 'able' 'according' 'action' 'added' 'already'
 'also' 'although' 'analyst' 'another' 'around' 'attack' 'award' 'away'
 'back' 'bank' 'bbc']


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, df['Category_Code'],
    test_size=0.15,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1891, 300)
Test shape: (334, 300)


In [12]:
import pickle

with open('features_train.pkl', 'wb') as f:
    pickle.dump(X_train, f)

with open('features_test.pkl', 'wb') as f:
    pickle.dump(X_test, f)